In [1]:
pip install tensorflow==2.10.0 tensorflow-hub==0.12.0

In [2]:
pip install --user transformers==4.44.2

In [3]:
pip install tensorflow-text

  Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached tensorflow-2.15.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached protobuf-4.25.5-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached tensorboard-2.15.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached tensorflow_estimator-2.15.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached keras-2.15.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached google_auth_oauthlib-1.2.1-py2.py3-none-any.whl.metadata (2.7 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
Using cached tensorflow-2.15.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (475.2 MB)
Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl (30 kB)
Using cached keras-2.15.0-py3-none-any.whl (1.7 MB)
Using cached protobuf-4.25.5-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
Using cached tensorboard-2.15.2-py3-none-any

In [9]:
#!/usr/bin/env python3

"""
question answer task 0 question answer
"""

import tensorflow_hub as hub
import tensorflow as tf
from transformers import BertTokenizer


def single_question_answer(question, reference):
    """
    question answer task
    """
    tokenizer = BertTokenizer.from_pretrained(
        'bert-large-uncased-whole-word-masking-finetuned-squad')
    model = hub.load("https://tfhub.dev/see--/bert-uncased-tf2-qa/1")

    """Tokenize the input question and reference text"""
    reference = tokenizer.tokenize(reference)
    reference = reference + ['[SEP]']
    reference_ids = tokenizer.convert_tokens_to_ids(reference)
    question = tokenizer.tokenize(question)
    question = ['[CLS]'] + question + ['[SEP]']
    question_tokens_ids = tokenizer.convert_tokens_to_ids(question)
    input_ids = question_tokens_ids + reference_ids
    input_mask = [1] * len(input_ids)
    input_types = [0] * len(question) + [1] * len(reference)

    """convert list to tensors and expand dimensions"""
    input_ids = tf.convert_to_tensor(input_ids, dtype=tf.int32)
    input_mask = tf.convert_to_tensor(input_mask, dtype=tf.int32)
    input_types = tf.convert_to_tensor(input_types, dtype=tf.int32)
    input_ids = tf.expand_dims(input_ids, axis=0)
    input_mask = tf.expand_dims(input_mask, axis=0)
    input_types = tf.expand_dims(input_types, axis=0)

    """Get the model's predictions"""
    outputs = model([input_ids, input_mask, input_types])
    start_logits = tf.argmax(outputs[0][0][1:-1]) + 1
    end_logits = tf.argmax(outputs[1][0][1:-1]) + 1

    """extract answer tokens and convert them to a string"""
    tokens = question + reference
    answer_tokens = tokens[start_logits: end_logits + 1]
    if len(answer_tokens) == 0:
        return None
    else:
        answer = tokenizer.convert_tokens_to_string(answer_tokens)
    return answer


In [5]:
#!/usr/bin/env python3

"""
question answer task 3 semantic search
"""

import tensorflow_hub as hub
import numpy as np
import os


def semantic_search(corpus_path, sentence):
    documents = [sentence]
    for filename in os.listdir(corpus_path):
        if filename.endswith(".md") is False:
            continue
        with open(corpus_path + "/" + filename, "r", encoding="utf-8") as f:
            documents.append(f.read())
    module_url = "https://tfhub.dev/google/universal-sentence-encoder-large/5"
    model = hub.load(module_url)
    embed = model(documents)
    corr = np.inner(embed, embed)
    close = np.argmax(corr[0, 1:])
    similarity = documents[close + 1]
    return similarity

In [10]:
""" task 4 multi reference question answer """
def question_answer(corpus_path):
    """answers questions from multiple reference texts:"""
    while True:
        question = input("Q: ").lower()

        if question in ["exit", "quit", "goodbye", "bye"]:
            print("A: Goodbye")
            exit()
        else:
            text = semantic_search(corpus_path, question)
            answer = single_question_answer(question, text)
            if answer is None:
                print("A: Sorry, I do not understand your question.")
            else:
                print("A:", answer)

In [11]:



question_answer('ZendeskArticles')

Q: what is holberton school


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

/root/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


A: a supportive learning space with clear expectations
Q: What does PLD stand for?
A: peer learning days
Q: how to join holberton school
A: Sorry, I do not understand your question.
Q: is there exams
A: there exams
Q: how many years is holberton school 
A: Sorry, I do not understand your question.
Q: hey
A: use the awesome search feature located in the lower right corner of your dashboard .


KeyboardInterrupt: Interrupted by user